# Vacunación COVID-19 y Mortalidad por Zona (Chile)

Notebook que implementa el plan definido en `plan.md`.

**Pregunta de investigación:** ¿Cómo influyó el porcentaje de vacunación contra COVID-19 en la mortalidad en las distintas zonas del país (Norte, Centro, Sur)?

**Modelo:** regresión lineal de panel (zona × semana)

```
y_zona,t = a + b_zona · X_zona,t + e_zona,t
```

* `y_zona,t` = tasa de fallecidos por 100.000 habitantes.
* `X_zona,t` = porcentaje de población vacunada acumulada en la zona.
* `b_zona` = coeficiente específico por zona (interacción Zona × X).

**ETL:** se usan **exclusivamente** tres archivos (sin escaneo recursivo):

| producto | archivo | columnas |
|---|---|---|
| 84 | `producto84/fallecidos_comuna_edad_totales_std.csv` | Region, Fecha, Total |
| 83 | `producto83/vacunacion_establecimiento_std.csv` | Establecimiento, Fecha, Dosis, Cantidad |
| 93 | `producto93/ContactosPorComuna.csv` | Region, Codigo comuna, Comuna, Poblacion |

Las tablas grandes se agregan **antes** de mapear zonas o reindexar a semanas (paso de optimización del plan).

## 1. Configuración y rutas

Raíces del proyecto, directorio de salida y rutas exactas a los tres productos.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata

import numpy as np
import pandas as pd

# Raíz del workspace: cwd o el primer ancestro que tenga plan.md
workspace = Path.cwd()
if not (workspace / "plan.md").exists():
    for parent in workspace.parents:
        if (parent / "plan.md").exists():
            workspace = parent
            break

covid_dir   = workspace / "data" / "COVID" / "datos-covid-19"
output_dir  = workspace / "outputs" / "COVID"
output_dir.mkdir(parents=True, exist_ok=True)

# Los tres únicos archivos permitidos por plan.md
source_files = {
    "deaths":      covid_dir / "producto84" / "fallecidos_comuna_edad_totales_std.csv",
    "vaccination": covid_dir / "producto83" / "vacunacion_establecimiento_std.csv",
    "population":  covid_dir / "producto93" / "ContactosPorComuna.csv",
}

missing = [p for p in source_files.values() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Faltan datos planificados: {missing}")

print(f"Workspace : {workspace}")
for key, path in source_files.items():
    print(f"  {key:11s} -> {path.relative_to(workspace)}")

## 2. Lectura de `plan.md` y listado de tareas

Se recupera el plan y se extraen sus encabezados y líneas de tarea como auditoría.

In [ ]:
plan_path = workspace / "plan.md"
plan_lines = [ln.strip() for ln in plan_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
task_lines = [
    ln for ln in plan_lines
    if ln.startswith(("-", "*", "+"))
    or re.match(r"^\d+[.)]", ln)
    or ln.startswith("#")
]
print(f"plan.md             : {plan_path.relative_to(workspace)}")
print(f"Líneas de tarea     : {len(task_lines)}")
print("\n".join("  " + ln for ln in task_lines))

## 3. Lectura con `usecols` y catálogo de datos

Solo se cargan las columnas necesarias de los tres productos (plan: optimización).
Se genera `data_catalog.csv` y `data_quality_report.csv` sobre los conjuntos crudos.

In [ ]:
data_catalog = pd.DataFrame([
    {"product": key, "file": str(path.relative_to(workspace)), "exists": path.exists()}
    for key, path in source_files.items()
])
data_catalog.to_csv(output_dir / "data_catalog.csv", index=False)
display(data_catalog)

# --- Carga solo de columnas necesarias ---
deaths_raw = pd.read_csv(source_files["deaths"], usecols=["Region", "Fecha", "Total"])
vac_raw    = pd.read_csv(source_files["vaccination"], usecols=["Establecimiento", "Fecha", "Dosis", "Cantidad"])
pop_raw    = pd.read_csv(source_files["population"], usecols=["Region", "Codigo comuna", "Comuna", "Poblacion"])

datasets = {
    "producto84_fallecidos": deaths_raw,
    "producto83_vacunacion": vac_raw,
    "producto93_poblacion":  pop_raw,
}
quality = pd.DataFrame([
    {
        "product": name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "duplicate_rows": int(frame.duplicated().sum()),
        "missing_cells": int(frame.isna().sum().sum()),
    }
    for name, frame in datasets.items()
])
quality.to_csv(output_dir / "data_quality_report.csv", index=False)
display(quality)
print({name: frame.shape for name, frame in datasets.items()})

## 4. Mapeo región → zona (supuesto #5 de plan.md)

La normalización quita acentos y puntuación; la asignación hace match exacto y luego
substring (necesario para `Del Libertador General Bernardo O'Higgins`, `Magallanes y la Antartica`).
El mapeo se construye **una sola vez** y se reutiliza para fallecidos y población.

In [ ]:
# Palabras clave por zona (en forma 'compacta' = sin acentos ni puntuación)
zone_aliases = {
    "Norte":  ["aricayparinacota", "tarapaca", "antofagasta", "atacama", "coquimbo"],
    "Centro": ["valparaiso", "metropolitana", "ohiggins", "libertador", "maule"],
    "Sur":    ["nuble", "biobio", "araucania", "losrios", "lagos", "aysen", "magallanes"],
}
keyword_zone = {kw: zone for zone, kws in zone_aliases.items() for kw in kws}
keyword_list = sorted(keyword_zone, key=len, reverse=True)

def strip_accents(value):
    value = unicodedata.normalize("NFKD", str(value))
    return "".join(ch for ch in value if not unicodedata.combining(ch)).lower()

def words(value):
    return re.sub(r"[^a-z0-9]+", " ", strip_accents(value)).strip()

def compact(value):
    return re.sub(r"[^a-z0-9]+", "", strip_accents(value))

def assign_zone(region):
    key = compact(region)
    if key in keyword_zone:
        return keyword_zone[key]
    return next((keyword_zone[kw] for kw in keyword_list if kw in key), None)

# Observadas en fallecidos y población
observed_regions = (
    pd.Index(deaths_raw["Region"].dropna().unique())
    .union(pd.Index(pop_raw["Region"].dropna().unique()))
)
region_zone = pd.Series({region: assign_zone(region) for region in observed_regions}, name="zone")
print("Región -> zona:")
print(region_zone.to_string())
unassigned = region_zone[region_zone.isna()]
if not unassigned.empty:
    raise ValueError(f"Regiones sin zona asignada: {list(unassigned.index)}")

## 5. Paso 1 — Fallecidos

- Se agrega `Total` por `Region + Fecha` **antes** de cualquier otra transformación
  (colapsa Comuna y Edad, plan paso 1).
- Se mapea la región a zona y se agrega a **semanas con inicio el lunes**.

In [ ]:
deaths = (
    deaths_raw.groupby(["Region", "Fecha"], as_index=False, sort=False)["Total"].sum()
    .rename(columns={"Region": "region", "Fecha": "date", "Total": "deaths"})
)
deaths["date"] = pd.to_datetime(deaths["date"], errors="coerce")
deaths["zone"] = deaths["region"].map(region_zone)

deaths_weekly = (
    deaths.dropna(subset=["date", "zone"])
    .assign(week=lambda df: df["date"].dt.to_period("W-MON").dt.start_time)
    .groupby(["zone", "week"], as_index=False, sort=False)["deaths"]
    .sum()
)
print(f"Fallecidos crudos : {len(deaths_raw):,} filas -> {len(deaths):,} (Región+Fecha)")
print(f"Panel semanal     : {deaths_weekly.shape}")
display(deaths_weekly.head())

## 6. Paso 2 — Población

- `Poblacion` se convierte a numérico.
- Se conserva un único valor por `Codigo comuna`.
- Se mapea región → zona y se suma por zona.
- Resumen por `Región + Zona` → `population_by_region_zone.csv`.

In [ ]:
population = (
    pop_raw.rename(columns={"Region": "region", "Comuna": "commune", "Poblacion": "population"})
    .copy()
)
population["population"] = pd.to_numeric(population["population"], errors="coerce")
population["zone"] = population["region"].map(region_zone)
population = population.dropna(subset=["zone", "population"])

# Un valor por comuna (preferencia: fila con población no nula)
population = population.sort_values("population", ascending=False).drop_duplicates(subset=["Codigo comuna"])

population_zone = population.groupby("zone", as_index=False, sort=False)["population"].sum()
population_region_zone = (
    population.groupby(["region", "zone"], as_index=False, sort=False)["population"].sum()
)
population_region_zone.to_csv(output_dir / "population_by_region_zone.csv", index=False)
display(population_region_zone)
print("Población por zona:")
display(population_zone)

## 7. Paso 3 — Vacunación

- Se lee `vacunacion_establecimiento_std.csv`.
- Si `Dosis` tiene varias categorías se conservan los registros de **primera dosis**;
  si tiene una sola categoría (como `Personas_vacunadas`) se conservan todos.
- Se agrega primero por `Establecimiento + Fecha`.

In [ ]:
dose_labels = vac_raw["Dosis"].astype("string").str.strip().dropna().unique()
print("Etiquetas de Dosis encontradas:", list(dose_labels))

if len(dose_labels) > 1:
    is_first = (
        vac_raw["Dosis"].astype(str).str.strip().str.lower()
        .str.contains(r"1|prim|unica", regex=True, na=False)
    )
    vac_use = vac_raw.loc[is_first].copy()
    print("Múltiples etiquetas -> se conservan los registros de primera dosis.")
else:
    vac_use = vac_raw.copy()
    print("Etiqueta única -> se conservan todos los registros (medida disponible).")

vaccination = (
    vac_use[["Establecimiento", "Fecha", "Cantidad"]]
    .groupby(["Establecimiento", "Fecha"], as_index=False, sort=False)["Cantidad"]
    .sum()
    .rename(columns={"Establecimiento": "establishment", "Fecha": "date", "Cantidad": "vaccinated"})
)
vaccination["vaccinated"] = pd.to_numeric(vaccination["vaccinated"], errors="coerce").fillna(0.0)

print(f"Vacunación cruda     : {len(vac_raw):,} filas")
print(f"Agregada Est+Fecha  : {len(vaccination):,} filas")
display(vaccination.head())

## 8. Paso 4 — Asignación geográfica de establecimientos

El archivo de vacunación no trae región, por lo que se normaliza el nombre del
establecimiento y se busca el nombre de la **comuna** dentro de él con una expresión
regular **compilada**. Cada establecimiento único se mapea a Norte, Centro o Sur.
Los no asignados quedan como auditoría en `vaccination_unmapped_establishments.csv`.

In [ ]:
# Comuna -> zona (nombre normalizado 'words')
commune_zone = {
    words(commune): zone
    for commune, zone in population[["commune", "zone"]].drop_duplicates().itertuples(index=False)
}
commune_keys = sorted((key for key in commune_zone if key), key=len, reverse=True)
commune_rx = re.compile(
    r"(?<![a-z])( +" + "|".join(re.escape(key) for key in commune_keys) + r")(?![a-z])"
)

def establishment_zone(establishment):
    match = commune_rx.search(" " + words(establishment))
    return commune_zone[match.group(1).strip()] if match else None

# Se calcula una sola vez por establecimiento único
estab_zone = {
    est: establishment_zone(est)
    for est in vaccination["establishment"].dropna().unique()
}
vaccination["zone"] = vaccination["establishment"].map(estab_zone)

unmapped = (
    vaccination.loc[vaccination["zone"].isna(), "establishment"]
    .drop_duplicates()
    .rename("unmapped_establishments")
)
unmapped.to_frame().to_csv(output_dir / "vaccination_unmapped_establishments.csv", index=False)

print(f"Establecimientos únicos : {len(estab_zone)}")
print(f"Asignación a zona       : {100 * vaccination['zone'].notna().mean():.2f}%")
print(f"No asignados (auditoría): {len(unmapped)}")
print("Ejemplos no asignados   :", unmapped.head().tolist())

# Agregar a semanas (inicio lunes) y acumular vacunados por zona
vaccination_weekly = (
    vaccination.dropna(subset=["date", "zone"])
    .assign(date=lambda df: pd.to_datetime(df["date"], errors="coerce"))
    .dropna(subset=["date"])
    .assign(week=lambda df: df["date"].dt.to_period("W-MON").dt.start_time)
    .groupby(["zone", "week"], as_index=False, sort=False)["vaccinated"]
    .sum()
    .sort_values(["zone", "week"])
)
vaccination_weekly["vaccinated_accumulated"] = (
    vaccination_weekly.groupby("zone")["vaccinated"].cumsum()
)
print(f"Vacunación semanal     : {vaccination_weekly.shape}")
display(vaccination_weekly.head())

## 9. Pasos 5 y 6 — Tasas y panel final zona × semana

- `tasa_mortalidad = fallecidos / poblacion * 100000`
- `pct_vacunacion = vacunados_acumulados / poblacion * 100`
- Merge de mortalidad y vacunación por `Zona + week`, agregando la población
  de la zona → panel listo para la regresión.

In [ ]:
panel = deaths_weekly.merge(vaccination_weekly, on=["zone", "week"], how="outer")
panel = panel.merge(population_zone, on="zone", how="left")
panel = panel.sort_values(["zone", "week"]).reset_index(drop=True)

num_cols = ["deaths", "vaccinated", "vaccinated_accumulated"]
panel[num_cols] = panel[num_cols].fillna(0.0)
panel = panel.dropna(subset=["population"])

panel["tasa_mortalidad"] = panel["deaths"] / panel["population"] * 100_000
panel["pct_vacunacion"]  = panel["vaccinated_accumulated"] / panel["population"] * 100

panel.to_csv(output_dir / "covid_zone_weekly_panel.csv", index=False)

print(f"Panel final           : {panel.shape}  (zonas × semanas)")
print("Semanas distintas por zona:")
print(panel.groupby("zone")["week"].nunique().to_string())
display(panel.head(8))

## 10. Regresión lineal por zona (interacción Zona × X)

Se estima el modelo panel por mínimos cuadrados:

```
tasa_mortalidad_z,t = a + b_norte·X + b_centro·X + b_sur·X + e
```

donde cada `b_zona` actúa solo sobre las observaciones de su zona.
Resultado → `zone_vaccination_ols.csv`.

In [ ]:
reg_data = panel.dropna(subset=["tasa_mortalidad", "pct_vacunacion"]).copy()
reg_data["intercept"] = 1.0
for zone in ["Norte", "Centro", "Sur"]:
    reg_data[f"vaccination_{zone.lower()}"] = (
        reg_data["pct_vacunacion"] * (reg_data["zone"] == zone)
    )

predictors = ["intercept"] + [f"vaccination_{z.lower()}" for z in ["Norte", "Centro", "Sur"]]
X = reg_data[predictors].to_numpy(dtype=float)
y = reg_data["tasa_mortalidad"].to_numpy(dtype=float)
coefficients, *_ = np.linalg.lstsq(X, y, rcond=None)

ols_results = pd.DataFrame({"term": predictors, "coefficient": coefficients})
ols_results.to_csv(output_dir / "zone_vaccination_ols.csv", index=False)

# R² para referencia
y_hat = X @ coefficients
ss_res = float(np.sum((y - y_hat) ** 2))
ss_tot = float(np.sum((y - y.mean()) ** 2))
r2 = 1 - ss_res / ss_tot

print(f"Observaciones usadas  : {len(reg_data)}")
print(f"R² (referencia)      : {r2:.4f}")
display(ols_results)

## 11. Salidas y validación

Se verifican los 7 archivos que plan.md pide en `outputs/COVID` y se escribe
`validation_report.json`.

In [ ]:
required_names = [
    "data_catalog.csv",
    "data_quality_report.csv",
    "covid_zone_weekly_panel.csv",
    "population_by_region_zone.csv",
    "vaccination_unmapped_establishments.csv",
    "zone_vaccination_ols.csv",
    "validation_report.json",
]

validation = {
    "plan_exists": plan_path.exists(),
    "products_used": list(source_files),
    "panel_rows": int(len(panel)),
    "panel_zones": sorted(panel["zone"].dropna().unique().tolist()),
    "population_by_zone": population_zone.to_dict(orient="records"),
    "unmapped_establishments_count": int(len(unmapped)),
    "required_outputs": {
        name: (output_dir / name).exists() for name in required_names
    },
}
validation["all_required_outputs_exist"] = all(validation["required_outputs"].values())
validation_path = output_dir / "validation_report.json"
validation_path.write_text(json.dumps(validation, indent=4, ensure_ascii=False), encoding="utf-8")

print(json.dumps(validation, indent=4, ensure_ascii=False))
if not validation["all_required_outputs_exist"]:
    raise RuntimeError("Faltan archivos de salida requeridos por plan.md")